# Powerlifting Meet Total Predictor
### Can we predict a lifter's next competition total from their history?

**Author:** Adam Kutrik  
**Dataset:** [OpenPowerlifting](https://openpowerlifting.gitlab.io/opl-csv/) — Public domain, updated regularly  
**Tools:** Python · pandas · scikit-learn · Matplotlib  

---

## Project Overview

This project explores whether a powerlifter's **next competition total** can be predicted using:
- Their **previous total** (kg)
- Their **bodyweight** at the previous meet (kg)
- The **number of days** between the two meets

The analysis follows a full data science workflow: raw data ingestion → cleaning & filtering → feature engineering → baseline modeling → linear regression — with honest reflection on what the model does and doesn't tell us.

### Research Question
> *Given a lifter's previous total, bodyweight, and time between meets, how accurately can we predict their next competition total?*

### Key Findings (Preview)
| Model | MAE (kg) | RMSE (kg) |
|---|---|---|
| Baseline (average change) | 22.79 | 42.65 |
| Linear Regression | 23.08 | 41.85 |

Linear regression offers a marginal improvement over simply adding the average gain. The dominant predictor is `PrevTotalKg` (coefficient ≈ 0.96), confirming that **a lifter's history is the strongest signal** — bodyweight fluctuation and rest time add relatively little predictive power for the total itself.

This finding motivated a follow-up question explored at the end: does bodyweight change + days between meets better explain *changes* in total rather than the total itself?

---
## 1. Setup & Data Loading

The dataset is sourced from [OpenPowerlifting](https://openpowerlifting.gitlab.io/opl-csv/), a comprehensive open-source record of powerlifting competitions worldwide. The raw file contains hundreds of columns covering athlete demographics, lift attempts, equipment type, and meet metadata.

We load it and immediately verify it read correctly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the full OpenPowerlifting dataset
# Download from: https://openpowerlifting.gitlab.io/opl-csv/
original = pd.read_csv("openpowerlifting.csv", low_memory=False)

print("Load status:", "Success!" if original is not None else "Failure!")
print("Shape:", original.shape)
print("\nColumn headers:")
print(list(original.columns))

FileNotFoundError: [Errno 2] No such file or directory: 'openpowerlifting-latest.csv'

---
## 2. Filtering & Cleaning

The raw dataset includes equipped lifting, partial events (bench-only, deadlift-only), and records with missing totals. We narrow to a consistent, comparable population:

- **Raw equipment only** — removes equipped (single-ply, multi-ply) lifters whose totals are not directly comparable
- **Full power (SBD)** — squat + bench + deadlift only; eliminates push-pull or single-lift events
- **Valid total required** — drops disqualifications (DQ) and bomb-outs (no total recorded)
- **Date must be parseable** — ensures temporal analysis is valid

We keep only the four columns needed for this analysis.

In [ ]:
# Filter to: Raw equipment, full power (SBD), valid total
database_v1 = original[
    (original["Equipment"] == "Raw") &
    (original["Event"] == "SBD")
].copy()

database_v1 = database_v1.dropna(subset=["TotalKg"])

# Retain only the columns needed for this analysis
cols = ["Name", "BodyweightKg", "TotalKg", "Date"]
database_v1 = database_v1[cols]

# Parse and validate dates
database_v1["Date"] = pd.to_datetime(database_v1["Date"], errors="coerce")
database_v1 = database_v1.dropna(subset=["Date"])

# Save filtered base dataset
database_v1.to_csv("database_v1.csv", index=False)

print(f"Rows after filtering: {len(database_v1):,}")
print(f"Unique lifters:       {database_v1['Name'].nunique():,}")
print("\nSample rows:")
print(database_v1.head())

---
## 3. Building the Comparison Dataset

To analyze *changes* in performance, we need at least two meets per lifter. We:

1. **Sort** each lifter's meets chronologically
2. **Filter** to lifters with 2+ meets (single-meet lifters can't produce a "previous" row)
3. **Shift** each column to create `PrevTotalKg`, `PrevBodyweightKg`, and `PrevDate`
4. **Calculate days** between consecutive meets
5. **Exclude** same-day entries (0 days) and gaps over 1,000 days — the latter removes cases where lifters return after a multi-year break, which skew time-based analysis

In [ ]:
# Sort chronologically within each lifter
database_v1 = database_v1.sort_values(by=["Name", "Date"], ascending=[True, True])

# Keep only lifters who have competed at least twice
database_v1 = database_v1[
    database_v1.groupby("Name")["Name"].transform("count") >= 2
]

print(f"Lifters with 2+ meets: {database_v1['Name'].nunique():,}")
print(f"Total meet entries:    {len(database_v1):,}")

# Verify no lifter slipped through with fewer than 2 meets
assert (database_v1["Name"].value_counts() < 2).sum() == 0, "Some lifters have fewer than 2 meets!"
print("\nTop 10 most active lifters:")
print(database_v1["Name"].value_counts().head(10))

In [ ]:
# Create "previous meet" columns using shift within each lifter group
database_v1["PrevTotalKg"]      = database_v1.groupby("Name")["TotalKg"].shift(1)
database_v1["PrevBodyweightKg"] = database_v1.groupby("Name")["BodyweightKg"].shift(1)
database_v1["PrevDate"]         = database_v1.groupby("Name")["Date"].shift(1)

# Calculate days between consecutive meets
database_v1["DaysSinceLastMeet"] = (database_v1["Date"] - database_v1["PrevDate"]).dt.days

# Exclude same-day meets (0 days) and very long gaps (>1000 days)
database_v1 = database_v1[
    (database_v1["DaysSinceLastMeet"] > 0) &
    (database_v1["DaysSinceLastMeet"] <= 1000)
]

# Drop NaN rows (first meet per lifter has no "previous" data)
compared_data = database_v1.dropna(
    subset=["PrevTotalKg", "PrevBodyweightKg", "DaysSinceLastMeet"]
).copy()

# Rename for clarity
compared_data = compared_data.rename(columns={"TotalKg": "CurrTotalKg"})

# Final column selection
final_cols = ["Name", "PrevTotalKg", "PrevBodyweightKg", "DaysSinceLastMeet", "CurrTotalKg"]
compared_data = compared_data[final_cols]

# Save comparison dataset
compared_data.to_csv("compared_data.csv", index=False)

print(f"Final comparison rows: {len(compared_data):,}")
print("\nSample:")
print(compared_data.head())

---
## 4. Exploratory Data Analysis (EDA)

Before modeling, we visualize the distributions and key relationships in the data. This helps us understand what we're working with and spot any anomalies.

In [ ]:
# Distribution of days between meets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(compared_data["DaysSinceLastMeet"], bins=40, edgecolor="black", color="steelblue")
axes[0].set_xlabel("Days Since Last Meet", fontsize=12)
axes[0].set_ylabel("Number of Meet Transitions", fontsize=12)
axes[0].set_title("Distribution of Time Between Meets", fontsize=13)
axes[0].axvline(compared_data["DaysSinceLastMeet"].median(), color="red",
                linestyle="--", label=f'Median: {compared_data["DaysSinceLastMeet"].median():.0f} days')
axes[0].legend()

# Previous vs current total scatter
axes[1].scatter(
    compared_data["PrevTotalKg"],
    compared_data["CurrTotalKg"],
    alpha=0.15, s=5, color="steelblue"
)
axes[1].set_xlabel("Previous Meet Total (kg)", fontsize=12)
axes[1].set_ylabel("Current Meet Total (kg)", fontsize=12)
axes[1].set_title("Previous vs. Current Meet Totals", fontsize=13)

# Identity line (if next == prev, points fall here)
lims = [compared_data["PrevTotalKg"].min(), compared_data["PrevTotalKg"].max()]
axes[1].plot(lims, lims, "r--", alpha=0.8, label="No change (y = x)")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("Dataset summary:")
print(compared_data[["PrevTotalKg", "PrevBodyweightKg", "DaysSinceLastMeet", "CurrTotalKg"]].describe().round(2))

In [ ]:
# Total change vs. time between meets
compared_data["TotalChangeKg"] = compared_data["CurrTotalKg"] - compared_data["PrevTotalKg"]
compared_data.to_csv("compared_data.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: days vs total change
axes[0].scatter(
    compared_data["DaysSinceLastMeet"],
    compared_data["TotalChangeKg"],
    alpha=0.15, s=5, color="steelblue"
)
axes[0].axhline(0, color="red", linestyle="--", linewidth=1.5, label="No change")
axes[0].set_xlabel("Days Since Last Meet", fontsize=12)
axes[0].set_ylabel("Change in Total (kg)", fontsize=12)
axes[0].set_title("Total Change vs. Time Between Meets", fontsize=13)
axes[0].legend()

# Distribution of total change
axes[1].hist(compared_data["TotalChangeKg"], bins=50, edgecolor="black", color="steelblue")
axes[1].axvline(compared_data["TotalChangeKg"].mean(), color="red",
                linestyle="--", label=f'Mean: {compared_data["TotalChangeKg"].mean():.2f} kg')
axes[1].set_xlabel("Change in Total (kg)", fontsize=12)
axes[1].set_ylabel("Frequency", fontsize=12)
axes[1].set_title("Distribution of Meet-to-Meet Total Change", fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_total_change.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Average change between meets: {compared_data['TotalChangeKg'].mean():.2f} kg")
print(f"Std deviation of change:      {compared_data['TotalChangeKg'].std():.2f} kg")

---
## 5. Baseline Model — Average Change

Before applying any machine learning, we establish a **naive baseline**: predict that every lifter's next total equals their previous total plus the average change across all lifters.

This is a meaningful benchmark — any more complex model should beat it to justify its added complexity.

**Baseline formula:**  
`PredictedTotal = PrevTotal + avg_change`  
where `avg_change = 10.65 kg` (the mean meet-to-meet improvement across ~357K transitions)

In [ ]:
# Baseline: predict every lifter gains the average amount
avg_change = compared_data["TotalChangeKg"].mean()
print(f"Average meet-to-meet total change: {avg_change:.2f} kg")

compared_data["PredictedTotalKg_Baseline"] = compared_data["PrevTotalKg"] + avg_change

# Evaluate baseline
mae_baseline  = (compared_data["CurrTotalKg"] - compared_data["PredictedTotalKg_Baseline"]).abs().mean()
rmse_baseline = ((compared_data["CurrTotalKg"] - compared_data["PredictedTotalKg_Baseline"]) ** 2).mean() ** 0.5

print(f"\nBaseline MAE:  {mae_baseline:.2f} kg")
print(f"Baseline RMSE: {rmse_baseline:.2f} kg")
print("\nInterpretation: On average, the naive baseline is off by ~22.8 kg per prediction.")

---
## 6. Linear Regression Model

We train a linear regression model using three features:
- `PrevTotalKg` — the lifter's total at their most recent meet
- `PrevBodyweightKg` — their bodyweight at the most recent meet
- `DaysSinceLastMeet` — days elapsed between the two meets

An 80/20 train-test split is used with `random_state=42` for reproducibility.

In [ ]:
# Features and target
X = compared_data[["PrevTotalKg", "PrevBodyweightKg", "DaysSinceLastMeet"]]
y = compared_data["CurrTotalKg"]

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples:     {len(X_test):,}")

# Fit model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nLinear Regression MAE:  {mae_lr:.2f} kg")
print(f"Linear Regression RMSE: {rmse_lr:.2f} kg")

In [ ]:
# Model coefficients — what does the model rely on?
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
}).sort_values("Coefficient", ascending=False)

print("Model Intercept:", round(model.intercept_, 2))
print("\nFeature Coefficients:")
print(coef_df.to_string(index=False))

# Visualize coefficients
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["steelblue" if c > 0 else "tomato" for c in coef_df["Coefficient"]]
bars = ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="black")
ax.set_xlabel("Coefficient Value", fontsize=12)
ax.set_title("Linear Regression Feature Coefficients", fontsize=13)
ax.axvline(0, color="black", linewidth=0.8)
for bar, val in zip(bars, coef_df["Coefficient"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("feature_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nKey insight: PrevTotalKg (≈0.96) dominates. Bodyweight and days add minimal signal.")

---
## 7. Model Comparison & Results

In [ ]:
# Side-by-side comparison
results = pd.DataFrame({
    "Model": ["Baseline (avg change)", "Linear Regression"],
    "MAE (kg)": [round(mae_baseline, 2), round(mae_lr, 2)],
    "RMSE (kg)": [round(rmse_baseline, 2), round(rmse_lr, 2)]
})

print(results.to_string(index=False))

# Residual plot for the regression model
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, residuals, alpha=0.15, s=5, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted Total (kg)", fontsize=12)
axes[0].set_ylabel("Residual (kg)", fontsize=12)
axes[0].set_title("Residuals vs. Predicted Values", fontsize=13)

axes[1].hist(residuals, bins=50, edgecolor="black", color="steelblue")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_xlabel("Residual (kg)", fontsize=12)
axes[1].set_ylabel("Frequency", fontsize=12)
axes[1].set_title("Distribution of Residuals", fontsize=13)

plt.tight_layout()
plt.savefig("residuals.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Reflection & Next Steps

### What the model tells us

Linear regression outperforms the naive baseline only marginally (RMSE: 41.85 vs. 42.65). The coefficient analysis reveals why:

| Feature | Coefficient |
|---|---|
| `PrevTotalKg` | 0.9617 |
| `PrevBodyweightKg` | 0.0967 |
| `DaysSinceLastMeet` | 0.0122 |

**Previous total carries almost all the predictive weight.** This makes intuitive sense — strength is highly consistent between meets for most lifters — but it also means this model is essentially a slight rescaling of the previous total. It's a solid *forecasting* tool, but it doesn't answer the more interesting question.

### The real question (follow-up work)

The original goal was to understand **what drives changes in total**, not just predict the absolute total. Given the coefficient findings, a more targeted follow-up analysis would:

1. Use **`TotalChangeKg`** as the target variable (instead of `CurrTotalKg`)
2. Engineer **`BodyweightChangeKg`** (difference between prev and curr bodyweight)  
3. Examine whether bodyweight fluctuation + rest time explains *improvement or decline*
4. Test non-linear models (Random Forest, XGBoost) that may capture interaction effects

### Limitations

- **Name collisions:** The dataset uses names to group athletes — lifters sharing a name may be conflated
- **Weight class changes:** A lifter cutting/gaining into a new class is treated the same as one staying put
- **Noise sources:** Injury, programming, coaching quality, and competition selection are unobserved
- **No sex split:** Male and female lifters have very different distributions and were pooled here

---
## Appendix: Environment & Reproducibility

```
pandas >= 1.5
numpy >= 1.23
scikit-learn >= 1.2
matplotlib >= 3.6
```

To reproduce:
1. Download the OpenPowerlifting CSV from https://openpowerlifting.gitlab.io/opl-csv/
2. Place it in the same directory as this notebook as `openpowerlifting-latest.csv`
3. Run all cells in order

Data files generated by this notebook:
- `database_v1.csv` — filtered raw data (Raw, SBD, valid total, 2+ meets)
- `compared_data.csv` — final comparison dataset with engineered features